In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 02 - Bronze -> Silver com MERGE
# MAGIC
# MAGIC Processa **somente as linhas do Bronze ainda nao processadas** (watermark `silver_issues` sobre `ingested_at`),
# MAGIC deduplica dentro do lote e aplica `MERGE`:
# MAGIC - chave nova -> INSERT
# MAGIC - chave existente **e** `updated_at` mais novo -> UPDATE
# MAGIC - chave existente com versao igual ou mais antiga -> ignora (protege contra dado fora de ordem e torna o reprocessamento idempotente)


In [0]:
%run ./00_common

In [0]:

run_id = new_run_id()
started = now_str()
entity = "silver_issues"

try:
    wm_str, _ = get_watermark(entity)

    bronze = spark.table(BRONZE)
    if wm_str:
        bronze = bronze.filter(F.col("ingested_at") > F.lit(wm_str).cast("timestamp"))

    # Fixa o teto do lote antes de processar, para nao perder linhas que cheguem durante o MERGE.
    max_ing = bronze.agg(F.max("ingested_at").cast("string")).first()[0]

    if max_ing is None:
        log_run(run_id, "silver_merge", entity, started, "SUCCESS", rows_in=0,
                detail="sem dados novos no bronze")
        dbutils.notebook.exit("Sem dados novos no Bronze.")

    batch_raw = bronze.filter(F.col("ingested_at") <= F.lit(max_ing).cast("timestamp"))
    rows_in = batch_raw.count()

    metrics = merge_silver(build_silver_batch(batch_raw))

    set_watermark(entity, max_ing, run_id)
    log_run(run_id, "silver_merge", entity, started, "SUCCESS",
            rows_in=rows_in, rows_inserted=metrics["inserted"], rows_updated=metrics["updated"],
            detail=f"bronze ate ingested_at={max_ing}")

    print(f"Linhas lidas do Bronze: {rows_in}")
    print(f"Inseridas: {metrics['inserted']} | Atualizadas: {metrics['updated']}")

except Exception as e:
    log_run(run_id, "silver_merge", entity, started, "FAILED", detail=f"{type(e).__name__}: {e}")
    raise